<a href="https://colab.research.google.com/github/Githubdiaries/VeinVision/blob/develop%2Fmodel/VeinVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New section

In [1]:
# STEP 1: Clone your VeinVision repo
!git clone https://github.com/Githubdiaries/VeinVision.git
%cd VeinVision

# STEP 2: Install requirements
!pip install opencv-python pillow numpy tqdm


Cloning into 'VeinVision'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 2), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (2/2), done.
/content/VeinVision


In [2]:
# Install tools
!pip install gdown opencv-python numpy pillow tqdm

# Clone CUBITAL
!git clone https://github.com/EdwinTSalcedo/CUBITAL.git
%cd CUBITAL

# BEST OPTION: Download the SMALL RESIZED DATASET directly in Colab
!gdown --fuzzy "https://drive.google.com/uc?id=1wEE4qBD/GIVE_RESIZED_DATASET_ID_HERE"

# Unzip dataset
!unzip -q *.zip -d dataset

print("Dataset downloaded & extracted successfully!")


Cloning into 'CUBITAL'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 302 (delta 6), reused 12 (delta 5), pack-reused 288 (from 2)
Receiving objects: 100% (302/302), 386.21 MiB | 13.94 MiB/s, done.
Resolving deltas: 100% (128/128), done.
Updating files: 100% (74/74), done.
/content/VeinVision/CUBITAL
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1wEE4qBD/GIVE_RESIZED_DATASET_ID_HERE

but Gdown can't. Please check connections and permissions.
unzip:  cannot find or open *.zip, *.zip.zip or *.zip.ZIP.

No zipfiles found.
Dataset downloaded & extracted successfully!


In [3]:
%cd /content/VeinVision/CUBITAL
!ls



/content/VeinVision/CUBITAL
cad	images	      LICENSE.md  README.md
edgeai	inference.py  notebooks   requirements.txt


In [4]:
# Stay in CUBITAL folder
%cd /content/VeinVision/CUBITAL

# Install gdown (once)
!pip install -q gdown

# Download the FULL CUBITAL dataset folder into "cubital_dataset"
!gdown --folder "https://drive.google.com/drive/folders/19DaK7T81qTxgBzirvBGdMyVUj8mOCcOx?usp=sharing" \
       -O cubital_dataset \
       --remaining-ok

# See what came inside
!ls cubital_dataset


/content/VeinVision/CUBITAL
Retrieving folder contents
Retrieving folder 1itPFfEvblAsP4ZEBzYGGOGrxNw2REc06 validation
Processing file 1DFMIGoDwKJVt-Zkyac3rTYtaRjiwGSe6 aleyda.pdf
Processing file 1WVbuqPF0EMxdY3TEDcnY62joKmCpEr13 danithsa.pdf
Processing file 1TUUeLNsWGJiE2GmwmNcuBPOInGjmCI23 paula.pdf
Processing file 1arohj9Cw1Bz9wFJOX6tlCGr2wltI_NAh validation_aleyda.zip
Processing file 1fIUnfYTjALhAAglakdxxZHMAXnX0PKJy validation_danithsa.zip
Processing file 1lEEEqO5_LPPF39ju2ZxoDvnRSEI2_4SN validation_paula.zip
Processing file 157LiXcIsWJFFOlsR9eNdagTiMYEgu2Z5 final_augmented_dataset.zip
Processing file 191uA9ErYRSXculIa3AXHqfBhXjd7O3St final_dataset.zip
Processing file 1-6hCFfxxFFCx1fuBaQODVqDVOiWPl42U square_augmented_dataset512x512.zip
Processing file 1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by square_final_dataset512x512.zip
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1DFMIGoDwK

In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))



CUDA available: True
Device: cuda


In [6]:
%cd /content/VeinVision/CUBITAL/cubital_dataset

!unzip -q square_final_dataset512x512.zip -d square_dataset

!ls square_dataset



/content/VeinVision/CUBITAL/cubital_dataset
square_dataset


In [7]:
%cd /content/VeinVision/CUBITAL/cubital_dataset/square_dataset
!ls


/content/VeinVision/CUBITAL/cubital_dataset/square_dataset
square_dataset


In [8]:
%cd /content/VeinVision/CUBITAL/cubital_dataset/square_dataset/square_dataset
!ls


/content/VeinVision/CUBITAL/cubital_dataset/square_dataset/square_dataset
dataset.csv  masks  nir_images	preprocessed_images


In [9]:
import os

print("Number of NIR images :", len(os.listdir("nir_images")))
print("Number of masks      :", len(os.listdir("masks")))


Number of NIR images : 2016
Number of masks      : 2016


In [10]:
# Go to VeinVision root
%cd /content/VeinVision

# 1) Create the folders that train.py expects
import os
os.makedirs("data/train_images", exist_ok=True)
os.makedirs("data/train_masks", exist_ok=True)

# 2) Define source (CUBITAL) and destination (VeinVision) paths
src_images = "/content/VeinVision/CUBITAL/cubital_dataset/square_dataset/square_dataset/nir_images"
src_masks  = "/content/VeinVision/CUBITAL/cubital_dataset/square_dataset/square_dataset/masks"

dst_images = "data/train_images"
dst_masks  = "data/train_masks"

# 3) Copy files over
import shutil

for fname in os.listdir(src_images):
    shutil.copy(os.path.join(src_images, fname),
                os.path.join(dst_images, fname))

for fname in os.listdir(src_masks):
    shutil.copy(os.path.join(src_masks, fname),
                os.path.join(dst_masks, fname))

# 4) Sanity check
print("Images in data/train_images:", len(os.listdir(dst_images)))
print("Masks  in data/train_masks :", len(os.listdir(dst_masks)))


/content/VeinVision
Images in data/train_images: 2016
Masks  in data/train_masks : 2016


In [11]:
%cd /content/VeinVision

import torch
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

!python train.py


/content/VeinVision
Using device: cuda
Traceback (most recent call last):
  File "/content/VeinVision/train.py", line 33, in <module>
    main()
  File "/content/VeinVision/train.py", line 28, in main
    loss = train_loop(train_loader, model, optimizer, loss_fn, device)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/VeinVision/train.py", line 9, in train_loop
    for imgs, masks in dataloader:
                       ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 732, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 788, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in 

In [12]:
%cd /content/VeinVision

%%writefile data_loader.py
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class VeinDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

        # High-quality, correct handling for NIR images (1-channel)
        self.img_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),  # force 1-channel
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])  # 1-channel stats
        ])

        # Masks: label maps, so NO normalization (we don't want to distort 0/1)
        self.mask_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()  # values stay in [0,1]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)

        return image, mask


def get_loaders(img_dir, mask_dir, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    dataset = VeinDataset(img_dir, mask_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return loader


/content/VeinVision


UsageError: Line magic function `%%writefile` not found.


In [13]:
%cd /content/VeinVision

code = r"""
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class VeinDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

        # HIGH QUALITY: 1-channel NIR normalization
        self.img_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

        # HIGH QUALITY: binary masks (0/1) → no normalization
        self.mask_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)

        return image, mask


def get_loaders(img_dir, mask_dir, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    dataset = VeinDataset(img_dir, mask_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return loader
"""

# Write to file
with open("data_loader.py", "w") as f:
    f.write(code)

print("Updated data_loader.py successfully.")


/content/VeinVision
Updated data_loader.py successfully.


In [14]:
%cd /content/VeinVision

import torch
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

!python train.py


/content/VeinVision
Using device: cuda
Traceback (most recent call last):
  File "/content/VeinVision/train.py", line 33, in <module>
    main()
  File "/content/VeinVision/train.py", line 28, in main
    loss = train_loop(train_loader, model, optimizer, loss_fn, device)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/VeinVision/train.py", line 12, in train_loop
    preds = model(imgs)
            ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/VeinVision/model_unet.py", line 37, in forward
    conv1 = self.dconv_down1(x)
            ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dis

In [1]:
%cd /content/VeinVision

code = r"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=1):
        super(UNet, self).__init__()

        # Encoder
        self.dconv_down1 = DoubleConv(n_channels, 64)
        self.dconv_down2 = DoubleConv(64, 128)
        self.dconv_down3 = DoubleConv(128, 256)
        self.dconv_down4 = DoubleConv(256, 512)

        self.maxpool = nn.MaxPool2d(2)

        # Bottleneck
        self.dconv_bottom = DoubleConv(512, 1024)

        # Decoder
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.dconv_up4 = DoubleConv(512 + 1024, 512)
        self.dconv_up3 = DoubleConv(256 + 512, 256)
        self.dconv_up2 = DoubleConv(128 + 256, 128)
        self.dconv_up1 = DoubleConv(64 + 128, 64)

        # 1-channel output for binary mask
        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        conv1 = self.dconv_down1(x)
        x = self.maxpool(conv1)

        conv2 = self.dconv_down2(x)
        x = self.maxpool(conv2)

        conv3 = self.dconv_down3(x)
        x = self.maxpool(conv3)

        conv4 = self.dconv_down4(x)
        x = self.maxpool(conv4)

        # Bottleneck
        x = self.dconv_bottom(x)

        # Decoder
        x = self.upsample(x)
        x = torch.cat([x, conv4], dim=1)
        x = self.dconv_up4(x)

        x = self.upsample(x)
        x = torch.cat([x, conv3], dim=1)
        x = self.dconv_up3(x)

        x = self.upsample(x)
        x = torch.cat([x, conv2], dim=1)
        x = self.dconv_up2(x)

        x = self.upsample(x)
        x = torch.cat([x, conv1], dim=1)
        x = self.dconv_up1(x)

        # raw logits, NO sigmoid here
        x = self.final_conv(x)
        return x
"""

with open("model_unet.py", "w") as f:
    f.write(code)

print("Updated model_unet.py successfully.")


[Errno 2] No such file or directory: '/content/VeinVision'
/content
Updated model_unet.py successfully.


In [2]:
%cd /content

import os

# 1) If VeinVision folder is missing (runtime reset), clone it again
if not os.path.isdir("VeinVision"):
    print("VeinVision folder not found. Cloning repo...")
    !git clone https://github.com/Githubdiaries/VeinVision.git

# 2) Go inside the repo
%cd /content/VeinVision

# 3) Show what is here
!pwd
!ls


/content
VeinVision folder not found. Cloning repo...
Cloning into 'VeinVision'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 2), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (2/2), done.
/content/VeinVision
/content/VeinVision
data_loader.py	model_unet.py  preprocessing.py  README.md  train.py


In [3]:
%cd /content/VeinVision

!pip install -q gdown

# make a folder to hold the dataset
!mkdir -p cubital_dataset

# download ONLY the 512x512 resized dataset (small but high quality)
!gdown "https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by" -O cubital_dataset/square_final_dataset512x512.zip

# check that it arrived
!ls cubital_dataset


/content/VeinVision
Downloading...
From (original): https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by
From (redirected): https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by&confirm=t&uuid=ce55604c-a234-4482-bb0b-7f89a94e3dde
To: /content/VeinVision/cubital_dataset/square_final_dataset512x512.zip
100% 480M/480M [00:03<00:00, 140MB/s]
square_final_dataset512x512.zip


In [4]:
%cd /content/VeinVision/cubital_dataset

!unzip -q square_final_dataset512x512.zip -d extracted

!ls extracted


/content/VeinVision/cubital_dataset
square_dataset


In [5]:
%cd /content/VeinVision/cubital_dataset/square_dataset
!ls


[Errno 2] No such file or directory: '/content/VeinVision/cubital_dataset/square_dataset'
/content/VeinVision/cubital_dataset
extracted  square_final_dataset512x512.zip


In [6]:
%cd /content/VeinVision/cubital_dataset/extracted
!ls


/content/VeinVision/cubital_dataset/extracted
square_dataset


In [7]:
%cd /content/VeinVision/cubital_dataset/extracted/square_dataset
!ls


/content/VeinVision/cubital_dataset/extracted/square_dataset
dataset.csv  masks  nir_images	preprocessed_images


In [8]:
import os, shutil

# Go to repo root
%cd /content/VeinVision

# 1) Create data folders if they don't exist
os.makedirs("data/train_images", exist_ok=True)
os.makedirs("data/train_masks", exist_ok=True)

# 2) Source paths (from CUBITAL dataset)
src_images = "/content/VeinVision/cubital_dataset/extracted/square_dataset/nir_images"
src_masks  = "/content/VeinVision/cubital_dataset/extracted/square_dataset/masks"

dst_images = "data/train_images"
dst_masks  = "data/train_masks"

# 3) Copy all images
for fname in os.listdir(src_images):
    shutil.copy(os.path.join(src_images, fname),
                os.path.join(dst_images, fname))

for fname in os.listdir(src_masks):
    shutil.copy(os.path.join(src_masks, fname),
                os.path.join(dst_masks, fname))

# 4) Sanity check
print("Images in data/train_images:", len(os.listdir(dst_images)))
print("Masks  in data/train_masks :", len(os.listdir(dst_masks)))


/content/VeinVision
Images in data/train_images: 2016
Masks  in data/train_masks : 2016


In [ ]:
%cd /content/VeinVision

code = r"""
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class VeinDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

        # 1-channel NIR image pipeline
        self.img_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])  # 1-channel stats
        ])

        # 1-channel binary mask pipeline (no normalize)
        self.mask_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()  # keep values in [0,1]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)

        return image, mask


def get_loaders(img_dir, mask_dir, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    dataset = VeinDataset(img_dir, mask_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return loader
"""

with open("data_loader.py", "w") as f:
    f.write(code)

print("Updated data_loader.py ✅")


In [9]:
%cd /content/VeinVision

code = r"""
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class VeinDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

        # 1-channel NIR image pipeline
        self.img_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])  # 1-channel stats
        ])

        # 1-channel binary mask pipeline (no normalize)
        self.mask_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()  # keep values in [0,1]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)

        return image, mask


def get_loaders(img_dir, mask_dir, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    dataset = VeinDataset(img_dir, mask_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return loader
"""

with open("data_loader.py", "w") as f:
    f.write(code)

print("Updated data_loader.py ✅")


/content/VeinVision
Updated data_loader.py ✅


In [10]:
%cd /content/VeinVision

code = r"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=1):
        super(UNet, self).__init__()

        # Encoder
        self.dconv_down1 = DoubleConv(n_channels, 64)
        self.dconv_down2 = DoubleConv(64, 128)
        self.dconv_down3 = DoubleConv(128, 256)
        self.dconv_down4 = DoubleConv(256, 512)

        self.maxpool = nn.MaxPool2d(2)

        # Bottleneck
        self.dconv_bottom = DoubleConv(512, 1024)

        # Decoder
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.dconv_up4 = DoubleConv(512 + 1024, 512)
        self.dconv_up3 = DoubleConv(256 + 512, 256)
        self.dconv_up2 = DoubleConv(128 + 256, 128)
        self.dconv_up1 = DoubleConv(64 + 128, 64)

        # 1-channel output for binary mask
        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        conv1 = self.dconv_down1(x)
        x = self.maxpool(conv1)

        conv2 = self.dconv_down2(x)
        x = self.maxpool(conv2)

        conv3 = self.dconv_down3(x)
        x = self.maxpool(conv3)

        conv4 = self.dconv_down4(x)
        x = self.maxpool(conv4)

        # Bottleneck
        x = self.dconv_bottom(x)

        # Decoder
        x = self.upsample(x)
        x = torch.cat([x, conv4], dim=1)
        x = self.dconv_up4(x)

        x = self.upsample(x)
        x = torch.cat([x, conv3], dim=1)
        x = self.dconv_up3(x)

        x = self.upsample(x)
        x = torch.cat([x, conv2], dim=1)
        x = self.dconv_up2(x)

        x = self.upsample(x)
        x = torch.cat([x, conv1], dim=1)
        x = self.dconv_up1(x)

        # raw logits, NO sigmoid here
        x = self.final_conv(x)
        return x
"""

with open("model_unet.py", "w") as f:
    f.write(code)

print("Updated model_unet.py ✅")


/content/VeinVision
Updated model_unet.py ✅


In [11]:
%cd /content/VeinVision

code = r"""
import torch
from torch import nn, optim
from data_loader import get_loaders
from model_unet import UNet


def train_loop(dataloader, model, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0

    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        # Forward pass
        logits = model(imgs)          # [B, 1, H, W] raw scores (no sigmoid)
        loss = loss_fn(logits, masks) # BCEWithLogitsLoss

        # Backward + update
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # All 2016 samples as train for now
    train_loader = get_loaders("data/train_images", "data/train_masks", batch_size=2)

    # 1-channel NIR in, 1-channel mask out
    model = UNet(n_channels=1, n_classes=1).to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Stable + standard for binary segmentation with logits
    loss_fn = nn.BCEWithLogitsLoss()

    epochs = 10  # we can increase later

    for epoch in range(epochs):
        loss = train_loop(train_loader, model, optimizer, loss_fn, device)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

    torch.save(model.state_dict(), "outputs/unet_vein.pth")
    print("Model saved to outputs/unet_vein.pth")


if __name__ == "__main__":
    main()
"""

with open("train.py", "w") as f:
    f.write(code)

print("Updated train.py ✅")


/content/VeinVision
Updated train.py ✅


In [1]:
%cd /content/VeinVision

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

!python train.py


[Errno 2] No such file or directory: '/content/VeinVision'
/content
CUDA available: True
Device: cuda
python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [2]:
%cd /content

# If VeinVision folder is gone (Colab reset), clone again
!git clone https://github.com/Githubdiaries/VeinVision.git

%cd VeinVision
!ls



/content
Cloning into 'VeinVision'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 2), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (2/2), done.
/content/VeinVision
data_loader.py	model_unet.py  preprocessing.py  README.md  train.py


In [3]:
%cd /content/VeinVision

code = r"""
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class VeinDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

        self.img_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

        self.mask_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)

        return image, mask


def get_loaders(img_dir, mask_dir, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    dataset = VeinDataset(img_dir, mask_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return loader
"""

with open("data_loader.py", "w") as f:
    f.write(code)

print("Updated data_loader.py ✅")


/content/VeinVision
Updated data_loader.py ✅


In [4]:
%cd /content/VeinVision

code = r"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=1):
        super(UNet, self).__init__()

        self.dconv_down1 = DoubleConv(n_channels, 64)
        self.dconv_down2 = DoubleConv(64, 128)
        self.dconv_down3 = DoubleConv(128, 256)
        self.dconv_down4 = DoubleConv(256, 512)

        self.maxpool = nn.MaxPool2d(2)

        self.dconv_bottom = DoubleConv(512, 1024)

        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.dconv_up4 = DoubleConv(512 + 1024, 512)
        self.dconv_up3 = DoubleConv(256 + 512, 256)
        self.dconv_up2 = DoubleConv(128 + 256, 128)
        self.dconv_up1 = DoubleConv(64 + 128, 64)

        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        conv1 = self.dconv_down1(x)
        x = self.maxpool(conv1)

        conv2 = self.dconv_down2(x)
        x = self.maxpool(conv2)

        conv3 = self.dconv_down3(x)
        x = self.maxpool(conv3)

        conv4 = self.dconv_down4(x)
        x = self.maxpool(conv4)

        x = self.dconv_bottom(x)

        x = self.upsample(x)
        x = torch.cat([x, conv4], dim=1)
        x = self.dconv_up4(x)

        x = self.upsample(x)
        x = torch.cat([x, conv3], dim=1)
        x = self.dconv_up3(x)

        x = self.upsample(x)
        x = torch.cat([x, conv2], dim=1)
        x = self.dconv_up2(x)

        x = self.upsample(x)
        x = torch.cat([x, conv1], dim=1)
        x = self.dconv_up1(x)

        return self.final_conv(x)
"""

with open("model_unet.py", "w") as f:
    f.write(code)

print("Updated model_unet.py ✅")


/content/VeinVision
Updated model_unet.py ✅


In [5]:
%cd /content/VeinVision

code = r"""
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from data_loader import VeinDataset
from model_unet import UNet
import os


def train_loop(dataloader, model, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0

    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Load full dataset
    full_dataset = VeinDataset("data/train_images", "data/train_masks")

    # Fast check: use first 200 samples
    indices = list(range(min(200, len(full_dataset))))
    subset = Subset(full_dataset, indices)
    train_loader = DataLoader(subset, batch_size=2, shuffle=True)

    model = UNet(n_channels=1, n_classes=1).to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    epochs = 3

    for epoch in range(epochs):
        loss = train_loop(train_loader, model, optimizer, loss_fn, device)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

    os.makedirs("outputs", exist_ok=True)
    torch.save(model.state_dict(), "outputs/unet_vein_quicktest.pth")
    print("Quick-test model saved to outputs/unet_vein_quicktest.pth")


if __name__ == "__main__":
    main()
"""

with open("train.py", "w") as f:
    f.write(code)

print("Updated train.py for quick test ✅")


/content/VeinVision
Updated train.py for quick test ✅


In [6]:
%cd /content/VeinVision

import os, shutil

# 1) Install gdown if not present
!pip install -q gdown

# 2) Make dataset folder
os.makedirs("cubital_dataset", exist_ok=True)

# 3) Download resized 512x512 dataset (small)
zip_path = "cubital_dataset/square_final_dataset512x512.zip"
if not os.path.isfile(zip_path):
    !gdown "https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by" -O {zip_path}

# 4) Unzip into 'extracted'
extract_dir = "cubital_dataset/extracted"
os.makedirs(extract_dir, exist_ok=True)
!unzip -q {zip_path} -d {extract_dir}

# 5) Paths inside extracted
src_root = os.path.join(extract_dir, "square_dataset")
src_images = os.path.join(src_root, "nir_images")
src_masks  = os.path.join(src_root, "masks")

print("Images source folder:", src_images)
print("Masks  source folder:", src_masks)
print("Images present:", len(os.listdir(src_images)) if os.path.isdir(src_images) else "NOT FOUND")
print("Masks  present:", len(os.listdir(src_masks)) if os.path.isdir(src_masks) else "NOT FOUND")

# 6) Prepare data/ folders
os.makedirs("data/train_images", exist_ok=True)
os.makedirs("data/train_masks", exist_ok=True)

# 7) Copy files
for fname in os.listdir(src_images):
    shutil.copy(os.path.join(src_images, fname),
                os.path.join("data/train_images", fname))

for fname in os.listdir(src_masks):
    shutil.copy(os.path.join(src_masks, fname),
                os.path.join("data/train_masks", fname))

print("Final counts:")
print("Images in data/train_images:", len(os.listdir("data/train_images")))
print("Masks  in data/train_masks :", len(os.listdir("data/train_masks")))


/content/VeinVision
Downloading...
From (original): https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by
From (redirected): https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by&confirm=t&uuid=73d5cedd-97c9-49ac-bdaa-e2da039c53fb
To: /content/VeinVision/cubital_dataset/square_final_dataset512x512.zip
100% 480M/480M [00:03<00:00, 143MB/s]
Images source folder: cubital_dataset/extracted/square_dataset/nir_images
Masks  source folder: cubital_dataset/extracted/square_dataset/masks
Images present: 2016
Masks  present: 2016
Final counts:
Images in data/train_images: 2016
Masks  in data/train_masks : 2016


In [7]:
%cd /content/VeinVision
!python train.py


/content/VeinVision
Using device: cuda
Epoch 1/3, Loss: 0.3663
Epoch 2/3, Loss: 0.2604
Epoch 3/3, Loss: 0.2144
Quick-test model saved to outputs/unet_vein_quicktest.pth


In [8]:
%cd /content/VeinVision

code = r"""
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from data_loader import VeinDataset
from model_unet import UNet
import os


def dice_coeff(preds, targets, eps=1e-6):
    # preds: logits -> convert to probs -> binarize
    probs = torch.sigmoid(preds)
    preds_bin = (probs > 0.5).float()

    intersection = (preds_bin * targets).sum(dim=(1, 2, 3))
    union = preds_bin.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    dice = (2 * intersection + eps) / union
    return dice.mean().item()


def train_one_epoch(loader, model, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def eval_one_epoch(loader, model, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = loss_fn(logits, masks)
            total_loss += loss.item()
            total_dice += dice_coeff(logits, masks)
    return total_loss / len(loader), total_dice / len(loader)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # Load full dataset
    full_dataset = VeinDataset("data/train_images", "data/train_masks")
    n_total = len(full_dataset)
    n_val = int(0.2 * n_total)
    n_train = n_total - n_val

    train_ds, val_ds = random_split(full_dataset, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=2, shuffle=False)

    model = UNet(n_channels=1, n_classes=1).to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    epochs = 15  # serious training now

    best_val_dice = 0.0
    os.makedirs("outputs", exist_ok=True)

    for epoch in range(epochs):
        train_loss = train_one_epoch(train_loader, model, optimizer, loss_fn, device)
        val_loss, val_dice = eval_one_epoch(val_loader, model, loss_fn, device)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Dice: {val_dice:.4f}")

        # save best model based on validation Dice
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), "outputs/unet_vein_best.pth")
            print(f"  👉 New best model saved (Dice={best_val_dice:.4f})")

    print("Training complete. Best Dice:", best_val_dice)


if __name__ == "__main__":
    main()
"""

with open("train_full.py", "w") as f:
    f.write(code)

print("Created train_full.py ✅")


/content/VeinVision
Created train_full.py ✅


In [1]:
%cd /content/VeinVision
!python train_full.py



[Errno 2] No such file or directory: '/content/VeinVision'
/content
python3: can't open file '/content/train_full.py': [Errno 2] No such file or directory


In [1]:
# === MASTER SETUP: RUN THIS AFTER EVERY COLAB RESET ===

import os, shutil

# 1) Go to /content and clone VeinVision fresh
%cd /content
if os.path.isdir("VeinVision"):
    shutil.rmtree("VeinVision")

!git clone https://github.com/Githubdiaries/VeinVision.git
%cd VeinVision
print("Now in:", os.getcwd())
print("Repo files:", os.listdir("."))

# 2) Install gdown (for dataset)
!pip install -q gdown

# 3) Overwrite data_loader.py with correct 1-channel version
data_loader_code = r"""
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class VeinDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

        self.img_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

        self.mask_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)

        return image, mask


def get_loaders(img_dir, mask_dir, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    dataset = VeinDataset(img_dir, mask_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return loader
"""
with open("data_loader.py", "w") as f:
    f.write(data_loader_code)
print("✔ data_loader.py written")

# 4) Overwrite model_unet.py with 1-channel UNet
model_unet_code = r"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=1):
        super(UNet, self).__init__()

        self.dconv_down1 = DoubleConv(n_channels, 64)
        self.dconv_down2 = DoubleConv(64, 128)
        self.dconv_down3 = DoubleConv(128, 256)
        self.dconv_down4 = DoubleConv(256, 512)

        self.maxpool = nn.MaxPool2d(2)

        self.dconv_bottom = DoubleConv(512, 1024)

        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.dconv_up4 = DoubleConv(512 + 1024, 512)
        self.dconv_up3 = DoubleConv(256 + 512, 256)
        self.dconv_up2 = DoubleConv(128 + 256, 128)
        self.dconv_up1 = DoubleConv(64 + 128, 64)

        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        conv1 = self.dconv_down1(x)
        x = self.maxpool(conv1)

        conv2 = self.dconv_down2(x)
        x = self.maxpool(conv2)

        conv3 = self.dconv_down3(x)
        x = self.maxpool(conv3)

        conv4 = self.dconv_down4(x)
        x = self.maxpool(conv4)

        x = self.dconv_bottom(x)

        x = self.upsample(x)
        x = torch.cat([x, conv4], dim=1)
        x = self.dconv_up4(x)

        x = self.upsample(x)
        x = torch.cat([x, conv3], dim=1)
        x = self.dconv_up3(x)

        x = self.upsample(x)
        x = torch.cat([x, conv2], dim=1)
        x = self.dconv_up2(x)

        x = self.upsample(x)
        x = torch.cat([x, conv1], dim=1)
        x = self.dconv_up1(x)

        return self.final_conv(x)
"""
with open("model_unet.py", "w") as f:
    f.write(model_unet_code)
print("✔ model_unet.py written")

# 5) Overwrite train_full.py (full training with val + Dice)
train_full_code = r"""
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from data_loader import VeinDataset
from model_unet import UNet
import os


def dice_coeff(preds, targets, eps=1e-6):
    probs = torch.sigmoid(preds)
    preds_bin = (probs > 0.5).float()
    intersection = (preds_bin * targets).sum(dim=(1, 2, 3))
    union = preds_bin.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    dice = (2 * intersection + eps) / union
    return dice.mean().item()


def train_one_epoch(loader, model, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def eval_one_epoch(loader, model, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = loss_fn(logits, masks)
            total_loss += loss.item()
            total_dice += dice_coeff(logits, masks)
    return total_loss / len(loader), total_dice / len(loader)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    full_dataset = VeinDataset("data/train_images", "data/train_masks")
    n_total = len(full_dataset)
    n_val = int(0.2 * n_total)
    n_train = n_total - n_val
    train_ds, val_ds = random_split(full_dataset, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=2, shuffle=False)

    model = UNet(n_channels=1, n_classes=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    epochs = 15
    best_val_dice = 0.0
    os.makedirs("outputs", exist_ok=True)

    for epoch in range(epochs):
        train_loss = train_one_epoch(train_loader, model, optimizer, loss_fn, device)
        val_loss, val_dice = eval_one_epoch(val_loader, model, loss_fn, device)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Dice: {val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), "outputs/unet_vein_best.pth")
            print(f"  👉 New best model saved (Dice={best_val_dice:.4f})")

    print("Training complete. Best Dice:", best_val_dice)


if __name__ == "__main__":
    main()
"""
with open("train_full.py", "w") as f:
    f.write(train_full_code)
print("✔ train_full.py written")

# 6) Download + prepare dataset (CUBITAL 512x512)
os.makedirs("cubital_dataset", exist_ok=True)
zip_path = "cubital_dataset/square_final_dataset512x512.zip"

if not os.path.isfile(zip_path):
    !gdown "https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by" -O {zip_path}

extract_dir = "cubital_dataset/extracted"
os.makedirs(extract_dir, exist_ok=True)
!unzip -q {zip_path} -d {extract_dir}

src_root = os.path.join(extract_dir, "square_dataset")
src_images = os.path.join(src_root, "nir_images")
src_masks  = os.path.join(src_root, "masks")

print("Images source folder:", src_images)
print("Masks  source folder:", src_masks)
print("Images present:", len(os.listdir(src_images)) if os.path.isdir(src_images) else "NOT FOUND")
print("Masks  present:", len(os.listdir(src_masks)) if os.path.isdir(src_masks) else "NOT FOUND")

# 7) Fill data/train_images and data/train_masks
os.makedirs("data/train_images", exist_ok=True)
os.makedirs("data/train_masks", exist_ok=True)

for fname in os.listdir("data/train_images"):
    os.remove(os.path.join("data/train_images", fname))
for fname in os.listdir("data/train_masks"):
    os.remove(os.path.join("data/train_masks", fname))

for fname in os.listdir(src_images):
    shutil.copy(os.path.join(src_images, fname),
                os.path.join("data/train_images", fname))

for fname in os.listdir(src_masks):
    shutil.copy(os.path.join(src_masks, fname),
                os.path.join("data/train_masks", fname))

print("Final counts:")
print("Images in data/train_images:", len(os.listdir("data/train_images")))
print("Masks  in data/train_masks :", len(os.listdir("data/train_masks")))
print("✔ MASTER SETUP COMPLETE")


/content
Cloning into 'VeinVision'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 2), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (2/2), done.
/content/VeinVision
Now in: /content/VeinVision
Repo files: ['train.py', 'data_loader.py', 'README.md', 'preprocessing.py', 'model_unet.py', '.git']
✔ data_loader.py written
✔ model_unet.py written
✔ train_full.py written
Downloading...
From (original): https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by
From (redirected): https://drive.google.com/uc?id=1-3M8C5L0zB3YdFvwDz4A7hj5jkdqO8by&confirm=t&uuid=ebe9cad6-cbc1-4373-b396-58c3dba5d38a
To: /content/VeinVision/cubital_dataset/square_final_dataset512x512.zip
100% 480M/480M [00:11<00:00, 40.5MB/s]
Images source folder: cubital_dataset/extracted/square_dataset/nir_images
Masks  source folder: cubital_dataset/extr

In [2]:
%cd /content/VeinVision
!python train_full.py


/content/VeinVision
Using device: cuda
Epoch 1/15 | Train Loss: 0.1691 | Val Loss: 0.0773 | Val Dice: 0.0000
  👉 New best model saved (Dice=0.0000)
Epoch 2/15 | Train Loss: 0.0472 | Val Loss: 0.0288 | Val Dice: 0.0000
Epoch 3/15 | Train Loss: 0.0192 | Val Loss: 0.0126 | Val Dice: 0.0000
Epoch 4/15 | Train Loss: 0.0093 | Val Loss: 0.0069 | Val Dice: 0.0000
Epoch 5/15 | Train Loss: 0.0055 | Val Loss: 0.0043 | Val Dice: 0.0000
Epoch 6/15 | Train Loss: 0.0037 | Val Loss: 0.0031 | Val Dice: 0.0000
Epoch 7/15 | Train Loss: 0.0028 | Val Loss: 0.0024 | Val Dice: 0.0000
Epoch 8/15 | Train Loss: 0.0022 | Val Loss: 0.0020 | Val Dice: 0.0000
Epoch 9/15 | Train Loss: 0.0020 | Val Loss: 0.0018 | Val Dice: 0.0000
Epoch 10/15 | Train Loss: 0.0018 | Val Loss: 0.0016 | Val Dice: 0.0000
Epoch 11/15 | Train Loss: 0.0017 | Val Loss: 0.0016 | Val Dice: 0.0000
Epoch 12/15 | Train Loss: 0.0016 | Val Loss: 0.0015 | Val Dice: 0.0000
Epoch 13/15 | Train Loss: 0.0016 | Val Loss: 0.0015 | Val Dice: 0.0000
Epoch 14

In [3]:
%cd /content/VeinVision

import torch
from torch.utils.data import DataLoader, random_split
from data_loader import VeinDataset
from model_unet import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 1) Rebuild same dataset + val split
full_dataset = VeinDataset("data/train_images", "data/train_masks")
n_total = len(full_dataset)
n_val = int(0.2 * n_total)
n_train = n_total - n_val
train_ds, val_ds = random_split(full_dataset, [n_train, n_val])

val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
imgs, masks = next(iter(val_loader))

print("Masks stats:")
print("  min:", masks.min().item())
print("  max:", masks.max().item())
print("  mean:", masks.mean().item())

# 2) Load best model
model = UNet(n_channels=1, n_classes=1).to(device)
state = torch.load("outputs/unet_vein_best.pth", map_location=device)
model.load_state_dict(state)
model.eval()

with torch.no_grad():
    logits = model(imgs.to(device))
    probs = torch.sigmoid(logits).cpu()

print("\nPred probs stats:")
print("  min:", probs.min().item())
print("  max:", probs.max().item())
print("  mean:", probs.mean().item())

# 3) How many pixels > 0.5?
preds_bin = (probs > 0.5).float()
print("\nPredicted foreground pixels per image:",
      preds_bin.sum(dim=(1,2,3)))
print("GT foreground pixels per image:",
      masks.sum(dim=(1,2,3)))


/content/VeinVision
Device: cuda
Masks stats:
  min: 0.0
  max: 0.007843137718737125
  mean: 0.00027076498372480273

Pred probs stats:
  min: 0.006708491127938032
  max: 0.1879884898662567
  mean: 0.07384492456912994

Predicted foreground pixels per image: tensor([0., 0., 0., 0.])
GT foreground pixels per image: tensor([87.4392, 62.0706, 63.1608, 71.2471])


In [4]:
%cd /content/VeinVision

code = r"""
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class VeinDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

        self.img_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

        self.mask_transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path)
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_transform(mask)

        # 🔥 binarize mask: any > 0 becomes 1
        mask = (mask > 0).float()

        return image, mask


def get_loaders(img_dir, mask_dir, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    dataset = VeinDataset(img_dir, mask_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return loader
"""

with open("data_loader.py", "w") as f:
    f.write(code)

print("Updated data_loader.py with binary masks ✅")


/content/VeinVision
Updated data_loader.py with binary masks ✅


In [5]:
%cd /content/VeinVision

code = r"""
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from data_loader import VeinDataset
from model_unet import UNet
import os


def soft_dice_coeff(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    dice = (2 * intersection + eps) / union
    return dice.mean().item()


def soft_dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps
    dice = (2 * intersection + eps) / union
    return 1.0 - dice.mean()


def train_one_epoch(loader, model, optimizer, bce_loss_fn, device):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)

        bce = bce_loss_fn(logits, masks)
        dice_l = soft_dice_loss(logits, masks)
        loss = bce + 0.5 * dice_l   # weight Dice a bit

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def eval_one_epoch(loader, model, bce_loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)

            bce = bce_loss_fn(logits, masks)
            dice_l = soft_dice_loss(logits, masks)
            loss = bce + 0.5 * dice_l

            total_loss += loss.item()
            total_dice += soft_dice_coeff(logits, masks)
    return total_loss / len(loader), total_dice / len(loader)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    full_dataset = VeinDataset("data/train_images", "data/train_masks")
    n_total = len(full_dataset)
    n_val = int(0.2 * n_total)
    n_train = n_total - n_val
    train_ds, val_ds = random_split(full_dataset, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=2, shuffle=False)

    model = UNet(n_channels=1, n_classes=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    bce_loss_fn = nn.BCEWithLogitsLoss()

    epochs = 15
    best_val_dice = 0.0
    os.makedirs("outputs", exist_ok=True)

    for epoch in range(epochs):
        train_loss = train_one_epoch(train_loader, model, optimizer, bce_loss_fn, device)
        val_loss, val_dice = eval_one_epoch(val_loader, model, bce_loss_fn, device)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Dice: {val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), "outputs/unet_vein_best_bce_dice.pth")
            print(f"  👉 New best model saved (Dice={best_val_dice:.4f})")

    print("Training complete. Best Dice:", best_val_dice)


if __name__ == "__main__":
    main()
"""

with open("train_full.py", "w") as f:
    f.write(code)

print("Updated train_full.py with BCE + Dice ✅")


/content/VeinVision
Updated train_full.py with BCE + Dice ✅


In [6]:
%cd /content/VeinVision
!python train_full.py


/content/VeinVision
Using device: cuda
Epoch 1/15 | Train Loss: 0.5219 | Val Loss: 0.3128 | Val Dice: 0.5402
  👉 New best model saved (Dice=0.5402)
Epoch 2/15 | Train Loss: 0.1950 | Val Loss: 0.1208 | Val Dice: 0.8176
  👉 New best model saved (Dice=0.8176)
Epoch 3/15 | Train Loss: 0.0834 | Val Loss: 0.0586 | Val Dice: 0.9117
  👉 New best model saved (Dice=0.9117)
Traceback (most recent call last):
  File "/content/VeinVision/train_full.py", line 103, in <module>
    main()
  File "/content/VeinVision/train_full.py", line 86, in main
    train_loss = train_one_epoch(train_loader, model, optimizer, bce_loss_fn, device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/VeinVision/train_full.py", line 41, in train_one_epoch
    total_loss += loss.item()
                  ^^^^^^^^^^^
KeyboardInterrupt


In [7]:
%cd /content/VeinVision
from google.colab import files
files.download("outputs/unet_vein_best_bce_dice.pth")


/content/VeinVision


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!ls /content/VeinVision/outputs


unet_vein_best_bce_dice.pth  unet_vein_best.pth
